In [8]:
# ==============================================================================
# SLIDE 4: IMPORTAÇÃO E INICIALIZAÇÃO DO MODELO TREINADO PARA INFERÊNCIA
# ==============================================================================
import cv2
from ultralytics import YOLO

# Carrega os pesos otimizados gerados durante o seu treinamento ('best.pt')
caminho_pesos = "runs/detect/libras_yolov8-2/weights/best.pt"
print(f"A carregar o modelo treinado de: {caminho_pesos}")
modelo = YOLO(caminho_pesos)


# ==============================================================================
# SLIDE 5: CAPTURA DE VÍDEO E CONFIGURAÇÃO DA WEBCAM
# ==============================================================================
# Inicializa a captura de vídeo utilizando a webcam padrão do computador (índice 0)
cap = cv2.VideoCapture(0)

# Define a largura e altura da resolução da captura para otimizar o desempenho
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    raise RuntimeError("Erro ao aceder à webcam. Verifique se a câmara está conectada.")

print("Iniciando reconhecimento de Libras em tempo real. Pressione 'q' para sair.")

from collections import deque

# Cria uma fila para armazenar as últimas 5 predições do modelo
historico_predicoes = deque(maxlen=5)

# ==============================================================================
# SLIDE 6: LAÇO DE PROCESSAMENTO DE QUADROS (INFERÊNCIA EM TEMPO REAL)
# ==============================================================================
while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        print("Aviso: Não foi possível ler o frame da webcam.")
        break

    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # O YOLO aceita 3 canais, então convertemos de volta para 3 canais em tons de cinza
    frame_processado = cv2.cvtColor(cinza, cv2.COLOR_GRAY2BGR)
    
    # O frame capturado ('frame') já é colorido (BGR). 
    # Enviamos diretamente para o YOLOv8 realizar a inferência de forma otimizada.
    #resultados = modelo(frame, conf=0.5, verbose=False)
    resultados = modelo.track(frame_processado, conf=0.4, persist=True, verbose=False)

    
    # O método .plot() aceita parâmetros para ajustar a espessura da linha e o tamanho da fonte,
    # o que melhora bastante a legibilidade visual quando há leve oscilação.
    frame_anotado = resultados[0].plot(line_width=2, font_size=1)
    
    # Exibe o resultado visual colorido e formatado em uma janela interativa
    cv2.imshow("Reconhecimento de Libras - YOLOv8", frame_anotado)

    # Interrompe o loop de execução ao pressionar a tecla 'q' no teclado
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# ==============================================================================
# SLIDE 7: LIBERAÇÃO DE RECURSOS E ENCERRAMENTO
# ==============================================================================
cap.release()
cv2.destroyAllWindows()
print("Sessão de reconhecimento encerrada e recursos de vídeo libertados.")

A carregar o modelo treinado de: runs/detect/libras_yolov8-2/weights/best.pt
Iniciando reconhecimento de Libras em tempo real. Pressione 'q' para sair.
Sessão de reconhecimento encerrada e recursos de vídeo libertados.
